In [1]:
#pyg or py data, dataset, dataloader?
# dictionary or Data as output of PROTACDataset?
    #What works with the data loader?
    #What works with pyg funcitons such as num_node_features?



# import torch dataset and dataloader

#from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
# Import from_smiles from pytorch geometric
from torch_geometric.utils import from_smiles
from torch_geometric.data import Data, Dataset, InMemoryDataset
from torch_geometric.loader import DataLoader
from all_functions import get_node_labels
from rdkit import Chem


    

class ProtacDataset(InMemoryDataset):

    def __init__(self, protac_df, transform=None):
        self.protac_df = protac_df
        self.protac_smiles = protac_df['PROTAC SMILES'].tolist()
        self.poi_smiles = protac_df['POI SMILES'].tolist()
        self.e3_smiles = protac_df['E3 SMILES'].tolist()
        self.node_boundaries = [get_node_labels(smiles, poi_smile, e3_smile) for smiles, poi_smile, e3_smile in zip(self.protac_smiles, self.poi_smiles, self.e3_smiles)]
        
        #self.substructure_labels = 

    def __len__(self):
        return len(self.protac_df)
    
    def __getitem__(self, idx):
        elem = from_smiles(self.protac_smiles[idx])
        elem["x"]=elem["x"].type(torch.float32)
        elem["edge_index"]=elem["edge_index"].type(torch.int64)
        elem["edge_attr"]=elem["edge_attr"].type(torch.float32)
        elem["node_boundaries"] = self.node_boundaries[idx]


        return elem


/home/knkn308/.conda/envs/env-protac-toolkit/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
protac_pub_trainset_df = pd.read_csv('../../data/augmented/protac_pub_testset_testing.csv')
protac_pub_testset_df = pd.read_csv('../../data/augmented/protac_pub_trainset_testing.csv')
train_set_pub = ProtacDataset(protac_df=protac_pub_trainset_df)

In [3]:
"""print(train_set_pub[0])
from torch_geometric.loader import DataLoader
train_loader = DataLoader(train_set_pub, batch_size=32, shuffle=True)
x = next(iter(train_loader))
print(x.node_boundaries)

len(train_set_pub[0].node_boundaries) #0 8, 1 7

print(x.node_boundaries[62+7])

#for i in range(20):
#    print(x.node_boundaries[i])"""

'print(train_set_pub[0])\nfrom torch_geometric.loader import DataLoader\ntrain_loader = DataLoader(train_set_pub, batch_size=32, shuffle=True)\nx = next(iter(train_loader))\nprint(x.node_boundaries)\n\nlen(train_set_pub[0].node_boundaries) #0 8, 1 7\n\nprint(x.node_boundaries[62+7])\n\n#for i in range(20):\n#    print(x.node_boundaries[i])'

In [4]:
#print(train_set_pub[0])
#train_set_pub[0].node_boundaries.dtype

In [26]:
from torch_geometric.nn import GraphConv
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


#optimizer = optim.Adam(model.parameters(), lr=0.001)

#GCNConv worked well before, with a test accuracy of around 0.93
#GraphConv: maybe converges faster?
class PROTACSplitter(torch.nn.Module):
    def __init__(self, node_feature_dim, edge_feature_dim):
        super(PROTACSplitter, self).__init__()
        self.conv1 = GraphConv(node_feature_dim, 16)# edge_dim=edge_feature_dim)
        self.linear_layer = torch.nn.Linear(16, 3)  # Output layer for 3 classes
    
    #def forward(self, data_batch):
    def forward(self, node_attr, edge_index):
        #data = batch['pyg_data']
        #print(f"data_batch.x: {data_batch.x}")
        #print(f"data_batch.edge_index: {data_batch.edge_index}")
        #z = self.conv1(data_batch.x, data_batch.edge_index)#, edge_attr)
        z = self.conv1(node_attr, edge_index)#, edge_attr)
        z = F.relu(z)
        y = self.linear_layer(z)
        return y
        
    def train_model(self, training_data, optimizer=optim.Adam, lr = 0.001, batch_size=32, criterion=nn.CrossEntropyLoss(), shuffle=True, epochs=5, get_substructure_acc=False):
        self.train()
        train_loader = DataLoader(training_data, batch_size=batch_size, shuffle=shuffle)
        optimizer=optimizer(self.parameters(), lr=lr)
        optimizer.zero_grad()

        avg_loss_list = []
        num_protacs = len(training_data)
        for epoch in range(epochs):
            total_loss = 0
            for train_data in train_loader:
                node_attr=train_data.x
                edge_index = train_data.edge_index
                raw_boundary_prediction = self.forward(node_attr, edge_index)             # "RuntimeError: mat1 and mat2 must have the same dtype"
                node_class_targets = train_data['node_boundaries']
                loss = criterion(raw_boundary_prediction, node_class_targets) 
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            if (epoch+1) % 5 == 0:
                print(f"Avg train loss (epoch: {epoch+1}): ", avg_loss_list[-1])
            avg_loss_list.append(total_loss / num_protacs) 

        return avg_loss_list

    def get_protac_data(self, protac_smiles):
        protac_data=from_smiles(protac_smiles)
        protac_data["x"]=protac_data["x"].type(torch.float32)
        protac_data["edge_index"]=protac_data["edge_index"].type(torch.int64)
        protac_data["edge_attr"]=protac_data["edge_attr"].type(torch.float32)
        return protac_data

    def predict_substructure(self, protac_smiles=None, protac_data=None): #Define protac_smiles or protac_data
        if protac_smiles is not None:
            #print(protac_smiles)
            #print(type(protac_smiles))
            protac_data = self.get_protac_data(protac_smiles) #from_smiles(protac_smiles) #
        else:
            protac_smiles = protac_data.smiles
        
        self.eval() 
        with torch.no_grad():
            raw_boundary_prediction = self.forward(protac_data.x, protac_data.edge_index)

            class_predictions = process_boundaries_to_substructure_labels_v2_single(protac_smiles, raw_boundary_prediction)

            probabilities = F.softmax(raw_boundary_prediction, dim=1)      
            #class_predictions = probabilities.argmax(dim=1)
        return class_predictions, probabilities

    #def exact_boundary_accuracy()

    #def grace_boundary_accuracy()

    #def substructure_accuracy()
        #process_boundaries_to_substructures(raw_boundary_prediction, node_class_targets)

    

In [27]:
node_feature_dim = train_set_pub.num_node_features
edge_feature_dim = train_set_pub.num_edge_features 
model = PROTACSplitter(node_feature_dim, edge_feature_dim)

In [14]:
train_set_pub[0].smiles

'CCC(NC(=O)C1CC(C(=O)CCCCCCCCCCN2CCC3(CC2)CC(C)N(c2ccc(C#N)c(Cl)c2)C3)CN1C(=O)C(NC(=O)C(C)NC)C(C)(C)C)c1ccccc1'

In [28]:
model.predict_substructure(protac_smiles='CCC(NC(=O)C1CC(C(=O)CCCCCCCCCCN2CCC3(CC2)CC(C)N(c2ccc(C#N)c(Cl)c2)C3)CN1C(=O)C(NC(=O)C(C)NC)C(C)(C)C)c1ccccc1') 

CCC(NC(=O)C1CC(C(=O)CCCCCCCCCCN2CCC3(CC2)CC(C)N(c2ccc(C#N)c(Cl)c2)C3)CN1C(=O)C(NC(=O)C(C)NC)C(C)(C)C)c1ccccc1
<class 'str'>
tensor([[-2.8973e+00, -2.7066e+00, -2.6844e-01],
        [-4.2940e+00, -4.2982e+00, -7.8128e-02],
        [-5.2769e+00, -5.3962e+00,  2.2736e-01],
        [-3.6893e+00, -3.9682e+00, -9.9267e-03],
        [-4.8967e+00, -5.3716e+00,  2.1431e-01],
        [-2.2713e+00, -2.5373e+00, -5.0401e-01],
        [-5.0056e+00, -5.3421e+00,  2.0431e-01],
        [-3.9788e+00, -4.0506e+00, -7.7351e-02],
        [-5.2794e+00, -5.4442e+00,  1.2527e-01],
        [-5.0736e+00, -5.4140e+00,  2.4852e-01],
        [-2.2713e+00, -2.5373e+00, -5.0401e-01],
        [-3.9399e+00, -4.0387e+00, -6.7231e-02],
        [-4.2940e+00, -4.2982e+00, -7.8128e-02],
        [-4.2940e+00, -4.2982e+00, -7.8128e-02],
        [-4.2940e+00, -4.2982e+00, -7.8128e-02],
        [-4.2940e+00, -4.2982e+00, -7.8128e-02],
        [-4.2940e+00, -4.2982e+00, -7.8128e-02],
        [-4.2940e+00, -4.2982e+00, -7.8128e

NameError: name 'process_boundaries_to_substructures_v2' is not defined

In [14]:

model.train_model(training_data=train_set_pub, batch_size=16, epochs = 20)
pass

Avg train loss (epoch: 5):  0.041148898778138335
Avg train loss (epoch: 10):  0.02705849872695075
Avg train loss (epoch: 15):  0.019445933677532053
Avg train loss (epoch: 20):  0.017197700010405645


In [43]:


model = PROTACSplitter()
for batch in DataLoader(train_set_pub, batch_size=32):
    # batch['protac_smiles'] = (batch_size, 1024)
    y_hat = model(batch)
    loss = loss_fn(y_hat, batch['node_boundaries'])
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Metric
    y_hat = torch.argmax(y_hat, dim=1) # (batch_size, num_nodes) -> (batch_size, 1)
    acc = (y_hat == batch['node_boundaries']).sum() / len(y_hat)


TypeError: PROTACSplitter.__init__() missing 2 required positional arguments: 'node_feature_dim' and 'edge_feature_dim'